In [1]:
import pandas as pd

In [2]:
return_average = pd.read_csv("03_total_avg.csv", header = 0)

In [3]:
return_average.rename(columns={"Unnamed: 0": "date"}, inplace=True)

In [4]:
return_average

,date,GRPO,PPO,SAC
0,2019-01-30,0.014351,0.024242,-0.005591
1,2019-02-28,0.017790,0.017790,-0.007340
2,2019-03-28,-0.006875,-0.006875,-0.011638
3,2019-04-26,0.022650,0.022650,0.015814
4,2019-05-24,-0.025741,-0.025741,-0.026127
...,...,...,...,...
71,2024-09-20,0.025878,0.014823,0.024577
72,2024-10-18,0.000756,0.014355,0.005265
73,2024-11-15,-0.023999,-0.029808,-0.035376
74,2024-12-16,0.062063,0.055430,0.052077


In [5]:
return_average = return_average[["date", "PPO"]]

In [6]:
return_average.rename(columns={"PPO": "return"}, inplace=True)

/tmp/ipykernel_1618243/2399321464.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  return_average.rename(columns={"PPO": "return"}, inplace=True)


In [7]:
return_average

,date,return
0,2019-01-30,0.024242
1,2019-02-28,0.017790
2,2019-03-28,-0.006875
3,2019-04-26,0.022650
4,2019-05-24,-0.025741
...,...,...
71,2024-09-20,0.014823
72,2024-10-18,0.014355
73,2024-11-15,-0.029808
74,2024-12-16,0.055430


In [8]:
q1 = return_average["return"].quantile(0.25)

In [9]:
bottom_25 = return_average[return_average["return"]<=q1]


In [10]:
bottom_25.describe()

,return
count,19.000000
mean,-0.044618
std,0.035333
min,-0.145655
25%,-0.047043
50%,-0.036309
75%,-0.023763
max,-0.012684


In [11]:
desc = bottom_25["return"].describe()
std_error = desc["std"] / (desc["count"] ** 0.5)

print(std_error)
print("skewness:", bottom_25["return"].skew()
        , "kurtosis:", bottom_25["return"].kurtosis())

0.00810593754544535
skewness: -1.793768968715894 kurtosis: 3.0124581241524235


In [8]:
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.sandwich_covariance import cov_hac

def newey_west_tstat(returns, maxlags=1):
    """
    논문 방식에 따른 Newey-West t-통계량 계산 함수
    입력:
        returns: 수익률 벡터 (list, np.array, pd.Series)
        maxlags: Newey-West 보정에 사용할 최대 시차
    출력:
        (평균 수익률, NW 표준오차, NW t-통계량)
    """
    returns = np.asarray(returns)
    T = len(returns)
    X = np.ones((T, 1))  # 상수항만 포함 (평균 추정)
    
    model = sm.OLS(returns, X).fit(cov_type='HAC', cov_kwds={'maxlags': maxlags})
    nw_cov = cov_hac(model, nlags=maxlags)
    # nw_se = np.sqrt(nw_cov[0, 0])
    # t_stat = model.params[0] / nw_se
    
    return model.params[0], model.bse[0], model.tvalues[0]


In [9]:

# 계산 실행
mean_return, nw_se, nw_tstat = newey_west_tstat(return_average["return"], maxlags=3)


In [10]:
from scipy.stats import t



# 단측 검정 (우측): P(T > t)
p_value = 1 - t.cdf(nw_tstat, df=len(return_average))

print(f"p-value = {p_value:.6f}")


p-value = 0.023660


In [11]:
nw_tstat

np.float64(2.016129445705101)

In [12]:
return_average

,date,return
0,2019-01-30,0.024242
1,2019-02-28,0.017790
2,2019-03-28,-0.006875
3,2019-04-26,0.022650
4,2019-05-24,-0.025741
...,...,...
71,2024-09-20,0.014823
72,2024-10-18,0.014355
73,2024-11-15,-0.029808
74,2024-12-16,0.055430


In [13]:
np.quantile(return_average['return'], 0.05)

np.float64(-0.05291829325)